# ML-03 — Frame Your Lane as an ML Task


## 1. My lane as an ML task (type)

My Refresh / Content Opportunity Scoring lane is a ranking and scoring task. The goal is to rank pages so a content strategist can review the highest-priority pages first when there are too many pages to inspect manually.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy
In the starter dataset, I will use a current-window decline proxy: a page is marked positive when trend_direction is "down". This is a defined proxy because it comes from the dataset’s trend rule, not a later observed outcome. For a stronger capstone target, I would use earlier data as features and measure whether a page continues to show decline in a separate future 30-day window.

In [5]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Makayla-Kelly/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

data_path = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

df["starter_decline_proxy"] = df["trend_direction"].eq("down").astype(int)

df[["content_id", "trend_direction", "starter_decline_proxy"]].head()

,content_id,trend_direction,starter_decline_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1


## 3. Success metric
My primary success metric is Precision@50. This measures how many of the top 50 recommended pages are genuinely declining according to the proxy or later observed outcome. Precision@50 fits the decision because a strategist has limited review time and needs the first pages in the queue to be useful.

## 4. The unit of analysis, as a real dataframe

The unit of analysis is one pseudonymized content item, or page, per row. I will focus on pages with at least 100 impressions because those pages have enough search visibility to make review more meaningful.

In [6]:
review_pages = df.loc[
    df["impressions_90d"].ge(100),
    [
        "content_id",
        "content_type",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "ctr",
        "days_since_last_update",
        "trend_direction",
        "starter_decline_proxy",
    ],
].copy()

print(f"Rows in reviewable-page slice: {len(review_pages):,}")
print("One row = one pseudonymized content page.")
review_pages.head(10)

Rows in reviewable-page slice: 22,006
One row = one pseudonymized content page.


,content_id,content_type,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,trend_direction,starter_decline_proxy
0,content_304f48230142,keyword article,3803,29,10.6,0.76,20,down,1
1,content_a1fb4e703a9e,keyword article,15320,7,20.3,0.05,25,down,1
2,content_9aa793d4d895,keyword article,12581,11,36.5,0.09,20,down,1
3,content_331d6c4de07b,keyword article,11751,58,6.2,0.49,22,stable,0
4,content_d99b7a2d90ca,keyword article,19140,24,44.0,0.13,14,down,1
5,content_d4084a4bc775,keyword article,3970,1,8.5,0.03,20,down,1
7,content_a63219c6e95a,keyword article,1724,1,21.2,0.06,22,stable,0
8,content_5e6c160719bc,keyword article,32574,29,46.0,0.09,20,down,1
9,content_c27558df2b0c,keyword article,1240,2,4.9,0.16,104,down,1
10,content_d8ee6cc6d642,keyword article,20919,324,2.2,1.55,104,stable,0


## 5. Why ML beats a fixed rule here

A fixed rule such as “declining pages with at least 100 impressions” still produces thousands of candidates, so it does not answer which pages should be reviewed first. A scoring approach can combine several signals, such as visibility, clicks, position, CTR, freshness, and engagement, to produce a more useful priority queue. However, it must be compared with a transparent fixed-rule baseline and must not use trend_direction or trend_pct as features when predicting the decline proxy, because those fields create leakage.